In [ ]:
%pip install --upgrade scikit-learn threadpoolctl

In [ ]:
%restart_python

In [ ]:
# ML notebook: Fraud detection with scikit-learn + MLflow

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import mlflow
import mlflow.sklearn

# 1. Read Gold table into Pandas
gold_df = spark.read.table("fraud_proj.gold_creditcard")
pdf = gold_df.toPandas()

print("Gold dataset shape:", pdf.shape)
print(pdf.head())

# 2. Features & label
X = pdf.drop(columns=["Class", "gold_ingest_ts"])
y = pdf["Class"]

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 4. Define model
lr = LogisticRegression(max_iter=1000, solver="liblinear")

# 5. Train model
lr.fit(X_train, y_train)

# 6. Predictions
y_pred = lr.predict(X_test)
y_proba = lr.predict_proba(X_test)[:, 1]

# 7. Evaluation
auc = roc_auc_score(y_test, y_proba)
print("ROC AUC:", auc)
print(classification_report(y_test, y_pred))

# 8. Log to MLflow
# mlflow.set_experiment("/Users/<your_email>/fraud_detection")
user = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{user}/fraud_detection")

with mlflow.start_run(run_name="sklearn_logreg") as run:
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_metric("roc_auc", auc)
    mlflow.sklearn.log_model(lr, "fraud_model")

print("✅ Model logged to MLflow")

# 9. Save predictions back into a table
preds_pdf = pd.DataFrame({
    "label": y_test.values,
    "prediction": y_pred,
    "probability": y_proba
})

# Convert back to Spark DataFrame
preds_sdf = spark.createDataFrame(preds_pdf)

preds_sdf.write.format("delta").mode("overwrite").saveAsTable("fraud_proj.gold_predictions")

# 10. Preview predictions
display(spark.table("fraud_proj.gold_predictions").limit(10))

In [ ]:
%pip install imbalanced-learn

In [ ]:
%restart_python

In [ ]:
# ML notebook: Fraud detection with RandomForest + MLflow

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import mlflow
import mlflow.sklearn

# 1. Read Gold table into Pandas
gold_df = spark.read.table("fraud_proj.gold_creditcard")
pdf = gold_df.toPandas()

print("Gold dataset shape:", pdf.shape)
print(pdf.head())

# 2. Features & label
X = pdf.drop(columns=["Class", "gold_ingest_ts", "Time", "Amount"])
y = pdf["Class"]
# 2. Features & label
# X = pdf.drop(columns=["Class", "gold_ingest_ts"])  # drop label + ingestion timestamp
# y = pdf["Class"]

# Drop/convert datetime columns if they exist
for col in X.select_dtypes(include=["datetime64[ns]"]).columns:
    print(f"Dropping datetime column: {col}")
    X = X.drop(columns=[col])

# Optional: drop identifier-like cols if present
# drop_cols = ["Time", "Amount"]  # already not great features
# X = X.drop(columns=[c for c in drop_cols if c in X.columns])

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 4. Define model (use class_weight to handle imbalance instead of SMOTE)
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

# 5. Train model
rf.fit(X_train, y_train)

# 6. Predictions
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

# 7. Evaluation
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print("ROC AUC:", roc_auc)
print("PR AUC:", pr_auc)
print(classification_report(y_test, y_pred))

# 8. Log to MLflow
user = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{user}/fraud_detection")

with mlflow.start_run(run_name="rf_classifier") as run:
    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 8)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.sklearn.log_model(rf, "fraud_rf_model")

print("✅ Model logged to MLflow")

# 9. Save predictions back into a Delta table
preds_pdf = pd.DataFrame({
    "label": y_test.values,
    "prediction": y_pred,
    "probability": y_proba
})

preds_sdf = spark.createDataFrame(preds_pdf)
preds_sdf.write.format("delta").mode("overwrite").saveAsTable("fraud_proj.gold_predictions")

# 10. Preview predictions
display(spark.table("fraud_proj.gold_predictions").limit(10))